In [ ]:
"""
analysis_OLS.py

This script merges the processed energy/building dataset with landscape metrics,
cleans the merged dataset, filters the neighborhood shapefile, performs
correlation and VIF diagnostics, and runs OLS regression models.

The script also keeps several supplementary data checks, including KWB filtering,
area/population threshold summaries, and urbanity-level counts.

Main workflow:
1. Set project structure
2. Merge energy/building and landscape files
3. Select variables
4. Produce summary statistics
5. Clean missing values
6. Filter shapefile according to retained neighborhoods
7. Rename variables
8. Compare Spearman correlation and VIF before/after removing PLAND_built
9. Run OLS regression excluding PLAND_built
10. Save correlation heatmaps
11. Run supplementary KWB checks
"""

import os
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from typing import Optional
from typing import List
warnings.filterwarnings("ignore", category=FutureWarning)


# ============================================================
# 0. PROJECT STRUCTURE
# ============================================================

BASE_DIR = os.getcwd()

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "output")
WEIGHT_DIR = os.path.join(BASE_DIR, "weights")
SRC_DIR = os.path.join(BASE_DIR, "src")
SHP_DIR = os.path.join(BASE_DIR, "shp")
MODEL_DIR = os.path.join(BASE_DIR, "model")
FIGURE_DIR = os.path.join(BASE_DIR, "figures")

for directory in [DATA_DIR, OUT_DIR, WEIGHT_DIR, SRC_DIR, SHP_DIR, MODEL_DIR, FIGURE_DIR]:
    os.makedirs(directory, exist_ok=True)

print("PROJECT STRUCTURE READY")
print("BASE_DIR:", BASE_DIR)


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def preview_df(df: pd.DataFrame, message: Optional[str] = None, n: int = 5) -> None:
    """Print a compact preview of a dataframe."""
    print("\n" + "=" * 80)
    if message:
        print(f" {message}")
        print("-" * 80)
    print(f"🔸 Shape: {df.shape}")
    print(f"🔸 Rows: {len(df)}, Columns: {len(df.columns)}")
    print(f"🔸 Column names (first 15): {df.columns.tolist()[:15]} ...")
    print("\n🔹 Preview (top rows):")
    print(df.head(n))
    print("=" * 80 + "\n")


def clean_missing(x):
    """Convert common missing-value strings into NaN."""
    if isinstance(x, str) and x.strip().lower() in {"", ".", "na", "n/a", "null", "nan", "--", "-"}:
        return np.nan
    return x


def significance_stars(p: float) -> str:
    """Return conventional significance stars."""
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.1:
        return "*"
    return ""


def calculate_vif(df_numeric: pd.DataFrame) -> pd.DataFrame:
    """Calculate VIF for a numeric dataframe."""
    X = df_numeric.copy()
    X = X.replace([np.inf, -np.inf], np.nan).dropna()

    X_const = sm.add_constant(X)
    vif = pd.DataFrame({
        "Variable": X.columns,
        "VIF": [variance_inflation_factor(X_const.values, i + 1) for i in range(len(X.columns))]
    })
    return vif


def save_spearman_heatmap(corr: pd.DataFrame, title: str, output_path: str) -> None:
    """Save a Spearman correlation heatmap."""
    plt.figure(figsize=(14, 16))
    sns.set_theme(style="white")

    sns.heatmap(
        corr,
        annot=True,
        fmt=".2f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.7}
    )

    plt.title(title, fontsize=16, pad=20)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f" Heatmap saved: {output_path}")




def summarize_thresholds(series: pd.Series, thresholds: List[float], label: str) -> pd.DataFrame:
    """Summarize cumulative percentages under selected thresholds."""
    s = pd.to_numeric(series, errors="coerce").dropna()

    rows = []
    for th in thresholds:
        pct = (s <= th).mean() * 100
        rows.append({"variable": label, "threshold": th, "percentage_leq_threshold": pct})
        print(f"{label} ≤ {th:>8}: {pct:6.2f}%")

    return pd.DataFrame(rows)


def summarize_percentiles(series: pd.Series, quantiles: List[float], label: str) -> pd.DataFrame:
    """Summarize selected percentiles."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    qv = s.quantile(quantiles)

    rows = []
    print(f"\n--- {label} percentiles ---")
    for q, v in qv.items():
        rows.append({"variable": label, "percentile": q, "value": v})
        print(f"{int(q * 100):>2}th percentile: {v:.2f}")

    return pd.DataFrame(rows)


# ============================================================
# 1. MERGE TWO INPUT FILES
# ============================================================

file_energy = os.path.join(OUT_DIR, "neighborhoods_energy_building_characteristics_merged_more.csv")
file_landscape = os.path.join(OUT_DIR, "form_merged_green_cleaned.csv")

if not os.path.exists(file_energy):
    raise FileNotFoundError(f"Energy/building file not found: {file_energy}")
if not os.path.exists(file_landscape):
    raise FileNotFoundError(f"Landscape file not found: {file_landscape}")

df_energy = pd.read_csv(file_energy, dtype={"buurtcode": str})
df_landscape = pd.read_csv(file_landscape, dtype={"buurtcode": str})

preview_df(df_energy, "Loaded energy/building dataset")
preview_df(df_landscape, "Loaded landscape dataset")

df_merged = df_energy.merge(df_landscape, on="buurtcode", how="inner")
preview_df(df_merged, "Step 1  Merged dataset")

merged_output = os.path.join(OUT_DIR, "neighborhoods_form_energy_merged.csv")
df_merged.to_csv(merged_output, index=False, encoding="utf-8-sig")
print(f"Merged file saved: {merged_output}")


# ============================================================
# 2. SELECT VARIABLES
# ============================================================

cols_keep = [
    "buurtcode",

    # Landscape composition/configuration variables
    "PLAND_built", "PLAND_water", "PLAND_Dense_Short", "PLAND_Dense_Tall",
    "PLAND_Sparse_Short", "PLAND_Sparse_Tall",
    "PD", "CONTAG", "SHDI",
    "AI_built", "AI_water", "AI_Dense_Short", "AI_Dense_Tall", "AI_Sparse_Short", "AI_Sparse_Tall",
    "IJI_built", "IJI_water", "IJI_Dense_Short", "IJI_Desne_Tall", "IJI_Dense_Tall",
    "IJI_Sparse_Short", "IJI_Sparse_Tall",

    # Building, demographic, and energy variables
    "bld_A_pct", "bev_dich", "p_1gezw", "p_bjj2k", "p_ink_hi", "ave_floor", "building_Density",
    "g_ele", "g_gas", "p_stadsv", "p_bewndw", "ec_inten_percapita", "g_hhgro", "ste_mvs", "ec_total"
]

existing_cols = [c for c in cols_keep if c in df_merged.columns]
missing_cols = [c for c in cols_keep if c not in df_merged.columns]

if missing_cols:
    print("\n Columns requested but not found in merged dataset:")
    print(missing_cols)

df_selected = df_merged[existing_cols].copy()
preview_df(df_selected, "Step 2  Selected dataset")

selected_output = os.path.join(OUT_DIR, "neighborhoods_form_energy_merged_selected.csv")
df_selected.to_csv(selected_output, index=False, encoding="utf-8-sig")
print(f"Selected file saved: {selected_output}")


# ============================================================
# 3. SUMMARY STATISTICS
# ============================================================

summary_df = df_selected.copy()
num_cols = [c for c in summary_df.columns if c != "buurtcode"]

for col in num_cols:
    summary_df[col] = pd.to_numeric(summary_df[col], errors="coerce")

# In this summary only, zeros are treated as missing values.
summary_no0 = summary_df[num_cols].replace(0, np.nan)

summary = pd.DataFrame({
    "count": summary_no0.count(),
    "mean": summary_no0.mean(),
    "median": summary_no0.median(),
    "min": summary_no0.min(),
    "max": summary_no0.max()
})

print("\n Step 3  Summary statistics preview:")
print(summary.head(10))

summary_output = os.path.join(OUT_DIR, "neighborhoods_form_energy_merged_selected_summary.csv")
summary.to_csv(summary_output, encoding="utf-8-sig")
print(f"Summary statistics saved: {summary_output}")


# ============================================================
# 4. CLEANING: MISSING VALUES + PLAND FILL + REMOVE ROWS
# ============================================================

df_cleaning = pd.read_csv(selected_output, dtype={"buurtcode": str})
preview_df(df_cleaning, "Step 4  Before cleaning")

df_cleaning = df_cleaning.applymap(clean_missing)

# Fill missing PLAND variables with 0.
# Interpretation: if a land-cover class is absent in a neighborhood, its PLAND is set to 0.
pland_cols = [c for c in df_cleaning.columns if c.startswith("PLAND_")]
pland_nan_counts = df_cleaning[pland_cols].isna().sum()
df_cleaning[pland_cols] = df_cleaning[pland_cols].fillna(0)

print("\n Number of NaN values filled with 0 in each PLAND_* variable:")
print(pland_nan_counts)

required_nonmissing_cols = [
    "bld_A_pct", "bev_dich", "p_1gezw", "p_bjj2k", "p_ink_hi", "ave_floor", "g_hhgro",
    "IJI_water", "AI_water", "IJI_built", "AI_built", "AI_Dense_Tall", "AI_Dense_Short", "PD", "SHDI"
]
required_nonmissing_cols = [c for c in required_nonmissing_cols if c in df_cleaning.columns]

missing_mask = df_cleaning[required_nonmissing_cols].isna().any(axis=1)
removed = df_cleaning[missing_mask].copy()
cleaned = df_cleaning[~missing_mask].copy()

print("\n🗑 Number of rows removed because of missing values in each required column:")
print(removed[required_nonmissing_cols].isna().sum())

preview_df(cleaned, "Step 4  Cleaned dataset")
preview_df(removed, "Step 4  Removed rows")

clean_output = os.path.join(MODEL_DIR, "form_energy_final_cleaned.csv")
removed_output = os.path.join(MODEL_DIR, "removed_rows_final.csv")

cleaned.to_csv(clean_output, index=False, encoding="utf-8-sig")
removed.to_csv(removed_output, index=False, encoding="utf-8-sig")
print(f"Cleaned file saved: {clean_output}")
print(f"Removed rows saved: {removed_output}")


# ============================================================
# 5. FILTER SHAPEFILE
# ============================================================

shapefile_path = os.path.join(DATA_DIR, "main_buurten selection.shp")

if os.path.exists(shapefile_path):
    gdf = gpd.read_file(shapefile_path)
    gdf["buurtcode"] = gdf["buurtcode"].astype(str)

    codes = cleaned["buurtcode"].astype(str).tolist()
    filtered_gdf = gdf[gdf["buurtcode"].isin(codes)].copy()

    preview_df(filtered_gdf, "Step 5  Filtered GeoDataFrame")

    row_count = len(filtered_gdf)
    shp_output = os.path.join(SHP_DIR, f"{row_count}_filtered_neighborhood_boundariesx.shp")
    filtered_gdf.to_file(shp_output, encoding="utf-8")

    print(f"Shapefile saved: {shp_output}")
else:
    print(f"\n Shapefile not found, skipped shapefile filtering: {shapefile_path}")


# ============================================================
# 6. RENAME COLUMNS
# ============================================================

df_model = cleaned.copy()

rename_dict = {
    "IJI_Desne_Tall": "IJI_Dense_Tall",
    "bev_dich": "pop_density",
    "p_bjj2k": "pre2000_pc",
    "p_ink_hi": "inc_hi_pc",
    "p_1gezw": "SF_pc",
    "bld_A_pct": "Aplus_pc",
    "g_hhgro": "hh_size"
}

df_model = df_model.rename(columns=rename_dict)
preview_df(df_model, "Step 6  Renamed dataset")

renamed_output = os.path.join(MODEL_DIR, "form_energy_final_cleaned_renamed.csv")
df_model.to_csv(renamed_output, index=False, encoding="utf-8-sig")
print(f"Renamed model file saved: {renamed_output}")


# ============================================================
# 7. SPEARMAN CORRELATION + VIF BEFORE/AFTER REMOVING PLAND_BUILT
# ============================================================

selected_cols_full = [
    "PLAND_built", "PLAND_water", "PLAND_Dense_Short", "PLAND_Dense_Tall", "PD", "SHDI",
    "AI_built", "AI_water", "AI_Dense_Short", "AI_Dense_Tall",
    "IJI_built", "IJI_water", "Aplus_pc", "pop_density", "SF_pc",
    "pre2000_pc", "inc_hi_pc", "hh_size", "ave_floor"
]

selected_cols_full = [c for c in selected_cols_full if c in df_model.columns]
selected_cols_without_pland_built = [c for c in selected_cols_full if c != "PLAND_built"]

# ---------- 7.1 Before removing PLAND_built ----------
df_num_full = df_model[selected_cols_full].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

corr_full = df_num_full.corr(method="spearman")
corr_full_output = os.path.join(MODEL_DIR, "spearman_corr_with_PLAND_built.csv")
corr_full.to_csv(corr_full_output, encoding="utf-8-sig")

vif_full = calculate_vif(df_num_full)
vif_full_output = os.path.join(MODEL_DIR, "vif_with_PLAND_built.csv")
vif_full.to_csv(vif_full_output, index=False, encoding="utf-8-sig")

print("\n Spearman correlation WITH PLAND_built saved:", corr_full_output)
print(" VIF WITH PLAND_built:")
print(vif_full)

heatmap_full_output = os.path.join(FIGURE_DIR, "spearman_heatmap_with_PLAND_built.png")
save_spearman_heatmap(
    corr=corr_full,
    title="Spearman Correlation Heatmap with PLAND_built",
    output_path=heatmap_full_output
)

# ---------- 7.2 After removing PLAND_built ----------
df_num_without = df_model[selected_cols_without_pland_built].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()

corr_without = df_num_without.corr(method="spearman")
corr_without_output = os.path.join(MODEL_DIR, "spearman_corr_without_PLAND_built.csv")
corr_without.to_csv(corr_without_output, encoding="utf-8-sig")

vif_without = calculate_vif(df_num_without)
vif_without_output = os.path.join(MODEL_DIR, "vif_without_PLAND_built.csv")
vif_without.to_csv(vif_without_output, index=False, encoding="utf-8-sig")

print("\n Spearman correlation WITHOUT PLAND_built saved:", corr_without_output)
print(" VIF WITHOUT PLAND_built:")
print(vif_without)

heatmap_without_output = os.path.join(FIGURE_DIR, "spearman_heatmap_without_PLAND_built.png")
save_spearman_heatmap(
    corr=corr_without,
    title="Spearman Correlation Heatmap without PLAND_built",
    output_path=heatmap_without_output
)

# ---------- 7.3 VIF comparison table ----------
vif_comparison = vif_full.rename(columns={"VIF": "VIF_with_PLAND_built"}).merge(
    vif_without.rename(columns={"VIF": "VIF_without_PLAND_built"}),
    on="Variable",
    how="outer"
)

vif_comparison_output = os.path.join(MODEL_DIR, "vif_comparison_with_without_PLAND_built.csv")
vif_comparison.to_csv(vif_comparison_output, index=False, encoding="utf-8-sig")

print("\n VIF comparison saved:", vif_comparison_output)
print(vif_comparison)

# ---------- 7.4 Optional direct comparison of PLAND_built correlations ----------
if "PLAND_built" in corr_full.columns:
    pland_built_corr = corr_full[["PLAND_built"]].sort_values("PLAND_built", ascending=False)
    pland_built_corr_output = os.path.join(MODEL_DIR, "correlation_of_PLAND_built_with_other_variables.csv")
    pland_built_corr.to_csv(pland_built_corr_output, encoding="utf-8-sig")
    print("\n Correlation of PLAND_built with other variables saved:", pland_built_corr_output)


# ============================================================
# 8. OLS REGRESSION EXCLUDING PLAND_BUILT
# ============================================================

df_ols = pd.read_csv(renamed_output, dtype={"buurtcode": str})

if "ec_inten_percapita" not in df_ols.columns:
    raise KeyError("Dependent variable 'ec_inten_percapita' not found in the model dataset.")

y = pd.to_numeric(df_ols["ec_inten_percapita"], errors="coerce").replace(0, np.nan)
y = np.log(y)

X = df_ols[selected_cols_without_pland_built].apply(pd.to_numeric, errors="coerce")

ols_data = pd.concat([y.rename("log_ec_inten_percapita"), X], axis=1)
ols_data = ols_data.replace([np.inf, -np.inf], np.nan).dropna()

y_clean = ols_data["log_ec_inten_percapita"]
X_clean = ols_data[selected_cols_without_pland_built]

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_clean),
    columns=selected_cols_without_pland_built,
    index=X_clean.index
)

X_scaled = sm.add_constant(X_scaled)
model = sm.OLS(y_clean, X_scaled).fit()

print("\n Step 8  OLS Regression Summary:")
print(model.summary())

ols_results = pd.DataFrame({
    "coef": model.params,
    "std_err": model.bse,
    "t_value": model.tvalues,
    "p_value": model.pvalues
})

conf = model.conf_int()
ols_results["ci_lower"] = conf[0]
ols_results["ci_upper"] = conf[1]
ols_results["stars"] = ols_results["p_value"].apply(significance_stars)
ols_results["coef_with_sig"] = ols_results["coef"].apply(lambda x: f"{x:.3f}") + ols_results["stars"]

print("\n Step 8 ✓ OLS coefficients preview:")
print(ols_results.head(10))

ols_output = os.path.join(MODEL_DIR, "ols_results_exclude_PLAND_built.csv")
ols_results.to_csv(ols_output, index=True, encoding="utf-8-sig")

model_summary_output = os.path.join(MODEL_DIR, "ols_summary_exclude_PLAND_built.txt")
with open(model_summary_output, "w", encoding="utf-8") as f:
    f.write(model.summary().as_text())

print(f"OLS results saved: {ols_output}")
print(f"OLS summary saved: {model_summary_output}")


# ============================================================
# 9. SUPPLEMENTARY CHECKS: FILTER KWB BY FINAL BUURTCODES
# ============================================================

kwb_file = os.path.join(DATA_DIR, "kwb-2022.xlsx")
filtered_kwb_output = os.path.join(OUT_DIR, "filtered_kwb_2022.xlsx")

if os.path.exists(kwb_file):
    df_kwb = pd.read_excel(kwb_file)
    df_final_codes = pd.read_csv(renamed_output, dtype={"buurtcode": str})

    print("\nfile_A / KWB column names:")
    print(df_kwb.columns.tolist())

    if "gwb_code" in df_kwb.columns:
        df_kwb["gwb_code"] = df_kwb["gwb_code"].astype(str).str.strip().str.upper()
        df_final_codes["buurtcode"] = df_final_codes["buurtcode"].astype(str).str.strip().str.upper()

        filtered_kwb = df_kwb[df_kwb["gwb_code"].isin(df_final_codes["buurtcode"])].copy()
        filtered_kwb.to_excel(filtered_kwb_output, index=False)

        print("\nKWB filtering check:")
        print("KWB original rows:", len(df_kwb))
        print("Final model buurtcode count:", df_final_codes["buurtcode"].nunique())
        print("Filtered KWB rows:", len(filtered_kwb))
        print(f"Filtered KWB saved: {filtered_kwb_output}")
    else:
        print(" Column 'gwb_code' not found in KWB file. Skipped KWB filtering.")
else:
    print(f"\n KWB file not found, skipped KWB filtering: {kwb_file}")


# ============================================================
# 10. SUPPLEMENTARY CHECKS: AREA AND POPULATION DISTRIBUTIONS
# ============================================================

if os.path.exists(kwb_file):
    df_kwb_check = pd.read_excel(kwb_file)

    threshold_results = []
    percentile_results = []

    if "a_lan_ha" in df_kwb_check.columns:
        print("\n Area threshold check: a_lan_ha")
        area_thresholds = [10, 25, 50, 100, 250, 300, 500, 1000, 2000]
        area_quantiles = [0.5, 0.75, 0.8, 0.9, 0.95, 0.99]
        threshold_results.append(summarize_thresholds(df_kwb_check["a_lan_ha"], area_thresholds, "a_lan_ha"))
        percentile_results.append(summarize_percentiles(df_kwb_check["a_lan_ha"], area_quantiles, "a_lan_ha"))
    else:
        print(" Column 'a_lan_ha' not found. Skipped area threshold check.")

    if "a_inw" in df_kwb_check.columns:
        print("\n Population threshold check: a_inw")
        pop_thresholds = [100, 250, 500, 1000, 2000, 3000, 5000, 10000, 20000, 50000]
        pop_quantiles = [0.5, 0.75, 0.8, 0.9, 0.95, 0.99]
        threshold_results.append(summarize_thresholds(df_kwb_check["a_inw"], pop_thresholds, "a_inw"))
        percentile_results.append(summarize_percentiles(df_kwb_check["a_inw"], pop_quantiles, "a_inw"))
    else:
        print(" Column 'a_inw' not found. Skipped population threshold check.")

    if threshold_results:
        threshold_summary = pd.concat(threshold_results, ignore_index=True)
        threshold_output = os.path.join(MODEL_DIR, "kwb_area_population_threshold_summary.csv")
        threshold_summary.to_csv(threshold_output, index=False, encoding="utf-8-sig")
        print(f"Threshold summary saved: {threshold_output}")

    if percentile_results:
        percentile_summary = pd.concat(percentile_results, ignore_index=True)
        percentile_output = os.path.join(MODEL_DIR, "kwb_area_population_percentile_summary.csv")
        percentile_summary.to_csv(percentile_output, index=False, encoding="utf-8-sig")
        print(f"Percentile summary saved: {percentile_output}")


# ============================================================
# 11. SUPPLEMENTARY CHECKS: POPULATION COVERAGE OF FILTERED KWB
# ============================================================

if os.path.exists(kwb_file) and os.path.exists(filtered_kwb_output):
    df_all = pd.read_excel(kwb_file)
    df_filtered = pd.read_excel(filtered_kwb_output)

    if "gwb_code" in df_all.columns and "gwb_code" in df_filtered.columns and "a_inw" in df_all.columns and "a_inw" in df_filtered.columns:
        df_all["gwb_code"] = df_all["gwb_code"].astype(str).str.strip().str.upper()
        df_filtered["gwb_code"] = df_filtered["gwb_code"].astype(str).str.strip().str.upper()

        all_buurt = df_all[df_all["gwb_code"].str.startswith("BU")].copy()
        filtered_buurt = df_filtered[df_filtered["gwb_code"].str.startswith("BU")].copy()

        all_buurt["a_inw"] = pd.to_numeric(all_buurt["a_inw"], errors="coerce")
        filtered_buurt["a_inw"] = pd.to_numeric(filtered_buurt["a_inw"], errors="coerce")

        sum_all = all_buurt["a_inw"].sum(skipna=True)
        sum_filtered = filtered_buurt["a_inw"].sum(skipna=True)
        percentage = (sum_filtered / sum_all * 100) if sum_all != 0 else np.nan

        population_coverage = pd.DataFrame({
            "dataset": ["kwb_2022_BU_only", "filtered_kwb_2022_BU_only"],
            "row_count": [len(all_buurt), len(filtered_buurt)],
            "a_inw_sum": [sum_all, sum_filtered]
        })
        population_coverage["percentage_of_total_population"] = [100, percentage]

        population_coverage_output = os.path.join(MODEL_DIR, "filtered_kwb_population_coverage.csv")
        population_coverage.to_csv(population_coverage_output, index=False, encoding="utf-8-sig")

        print("\n Population coverage check:")
        print(f"[kwb-2022.xlsx] BU rows: {len(all_buurt):,}")
        print(f"[filtered_kwb_2022.xlsx] BU rows: {len(filtered_buurt):,}")
        print(f"KWB BU a_inw sum: {sum_all:,.0f}")
        print(f"Filtered BU a_inw sum: {sum_filtered:,.0f}")
        print(f"Coverage percentage: {percentage:.2f}%")
        print(f"Population coverage saved: {population_coverage_output}")
    else:
        print(" Required columns for population coverage check not found. Skipped.")


# ============================================================
# 12. SUPPLEMENTARY CHECKS: STE_MVS COUNTS
# ============================================================

if os.path.exists(filtered_kwb_output):
    df_filtered_kwb = pd.read_excel(filtered_kwb_output)

    if "ste_mvs" in df_filtered_kwb.columns:
        df_filtered_kwb["ste_mvs"] = pd.to_numeric(df_filtered_kwb["ste_mvs"], errors="coerce")
        ste_counts = df_filtered_kwb["ste_mvs"].value_counts().sort_index()

        ste_counts_output = os.path.join(MODEL_DIR, "filtered_kwb_ste_mvs_counts.csv")
        ste_counts.to_csv(ste_counts_output, header=["count"], encoding="utf-8-sig")

        print("\n ste_mvs counts in filtered KWB:")
        print(ste_counts)
        print(f"ste_mvs counts saved: {ste_counts_output}")
    else:
        print(" Column 'ste_mvs' not found in filtered KWB. Skipped ste_mvs count.")


print("\n ALL ANALYSIS STEPS COMPLETED SUCCESSFULLY!")


PROJECT STRUCTURE READY
BASE_DIR: d:\1st\code\project

 Loaded energy/building dataset
--------------------------------------------------------------------------------
🔸 Shape: (11418, 42)
🔸 Rows: 11418, Columns: 42
🔸 Column names (first 15): ['buurtcode', 'building_count_total', 'ground_area', 'total_area', 'total_verblijfsobj', 'bld_A_total', 'bld_A_pct', 'hh_A_total', 'hh_A_pct', 'regio', 'gm_naam', 'gwb_code', 'a_inw', 'a_hh', 'g_hhgro'] ...

🔹 Preview (top rows):
    buurtcode  building_count_total    ground_area     total_area  \
0  BU00140000                   719   92254.899323  353627.414865   
1  BU00140001                   985  134661.430574  558236.349568   
2  BU00140002                   745   76029.135996  264826.079503   
3  BU00140003                   233   43256.649839  193759.266656   
4  BU00140005                  1086  118532.078604  406969.860717   

   total_verblijfsobj  bld_A_total  bld_A_pct  hh_A_total  hh_A_pct  \
0              2291.0         83.0      1